Part 1 – Problem Framing

The sales team is closing fewer deals, even though they have enough opportunities in the pipeline.
This means:
•	Opportunities are being created.
•	But fewer deals are converting to wins.
So the issue is likely in:
•	Sales process
•	Sales execution
•	Pricing or competition
•	Customer targeting
Where exactly is the drop happening?
Is deal quality changing?
Are sales cycles getting longer?
Are we losing more to competitors?
What actions will improve win rate?
The AI system should:
•	Identify where win rate dropped
•	Identify why it dropped
•	Recommend what action the CRO should take
To analyze this properly, I’ m assuming:
1.	CRM data is clean and accurate.
2.	All deals move through defined sales stages.
3.	Win/Loss reasons are recorded.
4.	We have at least 6–12 months of historical data.
5.	Pipeline volume being “healthy” means:
	    Similar or higher number of opportunities than before.
6.	No major external market shock (like regulation change or pricing overhaul).

Part 2 – Data Exploration & Insights

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv('skygeni_sales_data.csv')

In [4]:
df.head()

,deal_id,created_date,closed_date,sales_rep_id,industry,region,product_type,lead_source,deal_stage,deal_amount,sales_cycle_days,outcome
0,D00001,2023-11-24,2023-12-15,rep_22,SaaS,North America,Enterprise,Referral,Qualified,4253,21,Won
1,D00002,2023-01-17,2023-01-27,rep_7,SaaS,India,Core,Referral,Closed,3905,10,Won
2,D00003,2023-10-29,2023-12-10,rep_5,HealthTech,APAC,Core,Inbound,Proposal,10615,42,Lost
3,D00004,2023-07-14,2023-08-02,rep_18,FinTech,India,Core,Partner,Negotiation,4817,19,Won
4,D00005,2024-02-29,2024-05-26,rep_2,HealthTech,APAC,Core,Outbound,Qualified,45203,87,Lost


In [5]:
df.tail()

,deal_id,created_date,closed_date,sales_rep_id,industry,region,product_type,lead_source,deal_stage,deal_amount,sales_cycle_days,outcome
4995,D04996,2023-10-17,2023-12-03,rep_13,Ecommerce,North America,Enterprise,Partner,Closed,2586,47,Lost
4996,D04997,2023-11-11,2023-12-09,rep_20,FinTech,APAC,Core,Referral,Closed,10589,28,Lost
4997,D04998,2023-10-19,2023-10-27,rep_24,FinTech,North America,Core,Inbound,Negotiation,57434,8,Won
4998,D04999,2023-03-12,2023-05-18,rep_21,SaaS,APAC,Enterprise,Inbound,Proposal,50717,67,Won
4999,D05000,2023-12-22,2024-03-25,rep_5,HealthTech,North America,Core,Outbound,Demo,10524,94,Won


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   deal_id           5000 non-null   object
 1   created_date      5000 non-null   object
 2   closed_date       5000 non-null   object
 3   sales_rep_id      5000 non-null   object
 4   industry          5000 non-null   object
 5   region            5000 non-null   object
 6   product_type      5000 non-null   object
 7   lead_source       5000 non-null   object
 8   deal_stage        5000 non-null   object
 9   deal_amount       5000 non-null   int64 
 10  sales_cycle_days  5000 non-null   int64 
 11  outcome           5000 non-null   object
dtypes: int64(2), object(10)
memory usage: 468.9+ KB


In [7]:
df.isna().sum()

deal_id             0
created_date        0
closed_date         0
sales_rep_id        0
industry            0
region              0
product_type        0
lead_source         0
deal_stage          0
deal_amount         0
sales_cycle_days    0
outcome             0
dtype: int64

In [8]:
df['created_date'] = pd.to_datetime(df['created_date'])
df['closed_date'] = pd.to_datetime(df['closed_date'])

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   deal_id           5000 non-null   object        
 1   created_date      5000 non-null   datetime64[ns]
 2   closed_date       5000 non-null   datetime64[ns]
 3   sales_rep_id      5000 non-null   object        
 4   industry          5000 non-null   object        
 5   region            5000 non-null   object        
 6   product_type      5000 non-null   object        
 7   lead_source       5000 non-null   object        
 8   deal_stage        5000 non-null   object        
 9   deal_amount       5000 non-null   int64         
 10  sales_cycle_days  5000 non-null   int64         
 11  outcome           5000 non-null   object        
dtypes: datetime64[ns](2), int64(2), object(8)
memory usage: 468.9+ KB


In [10]:
industries = df['industry'].unique()
industries

array(['SaaS', 'HealthTech', 'FinTech', 'EdTech', 'Ecommerce'],
      dtype=object)

In [11]:
regions = df['region'].unique()
regions

array(['North America', 'India', 'APAC', 'Europe'], dtype=object)

In [12]:
products = df['product_type'].unique()
products

array(['Enterprise', 'Core', 'Pro'], dtype=object)

In [13]:
deal_stages = df['deal_stage'].unique()
deal_stages

array(['Qualified', 'Closed', 'Proposal', 'Negotiation', 'Demo'],
      dtype=object)

In [14]:
lead_source = df['deal_stage'].unique()
lead_source

array(['Qualified', 'Closed', 'Proposal', 'Negotiation', 'Demo'],
      dtype=object)

In [15]:
df['total_days'] = (df['closed_date'] - df['created_date']).dt.days

In [16]:
df.head()

,deal_id,created_date,closed_date,sales_rep_id,industry,region,product_type,lead_source,deal_stage,deal_amount,sales_cycle_days,outcome,total_days
0,D00001,2023-11-24,2023-12-15,rep_22,SaaS,North America,Enterprise,Referral,Qualified,4253,21,Won,21
1,D00002,2023-01-17,2023-01-27,rep_7,SaaS,India,Core,Referral,Closed,3905,10,Won,10
2,D00003,2023-10-29,2023-12-10,rep_5,HealthTech,APAC,Core,Inbound,Proposal,10615,42,Lost,42
3,D00004,2023-07-14,2023-08-02,rep_18,FinTech,India,Core,Partner,Negotiation,4817,19,Won,19
4,D00005,2024-02-29,2024-05-26,rep_2,HealthTech,APAC,Core,Outbound,Qualified,45203,87,Lost,87


In [17]:
df['close_quarter'] = df['closed_date'].dt.quarter

In [18]:
df.head()

,deal_id,created_date,closed_date,sales_rep_id,industry,region,product_type,lead_source,deal_stage,deal_amount,sales_cycle_days,outcome,total_days,close_quarter
0,D00001,2023-11-24,2023-12-15,rep_22,SaaS,North America,Enterprise,Referral,Qualified,4253,21,Won,21,4
1,D00002,2023-01-17,2023-01-27,rep_7,SaaS,India,Core,Referral,Closed,3905,10,Won,10,1
2,D00003,2023-10-29,2023-12-10,rep_5,HealthTech,APAC,Core,Inbound,Proposal,10615,42,Lost,42,4
3,D00004,2023-07-14,2023-08-02,rep_18,FinTech,India,Core,Partner,Negotiation,4817,19,Won,19,3
4,D00005,2024-02-29,2024-05-26,rep_2,HealthTech,APAC,Core,Outbound,Qualified,45203,87,Lost,87,2


In [19]:
df['is_won'] = np.where(df['outcome'] == 'Won', 1, 0)

In [20]:
df.describe()

,created_date,closed_date,deal_amount,sales_cycle_days,total_days,close_quarter,is_won
count,5000,5000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,2023-08-14 06:10:56.640000,2023-10-17 00:13:32.160000,26286.492800,63.751800,63.751800,2.356200,0.452600
min,2023-01-01 00:00:00,2023-01-11 00:00:00,2002.000000,7.000000,7.000000,1.000000,0.000000
25%,2023-04-22 00:00:00,2023-06-26 00:00:00,6611.000000,35.750000,35.750000,1.000000,0.000000
50%,2023-08-14 00:00:00,2023-10-15 00:00:00,14171.500000,64.000000,64.000000,2.000000,0.000000
75%,2023-12-06 00:00:00,2024-02-08 00:00:00,39062.250000,92.000000,92.000000,3.000000,1.000000
max,2024-03-26 00:00:00,2024-07-20 00:00:00,100000.000000,120.000000,120.000000,4.000000,1.000000
std,NaN,NaN,27689.230136,32.731405,32.731405,1.080535,0.497798


In [21]:
df.head()

,deal_id,created_date,closed_date,sales_rep_id,industry,region,product_type,lead_source,deal_stage,deal_amount,sales_cycle_days,outcome,total_days,close_quarter,is_won
0,D00001,2023-11-24,2023-12-15,rep_22,SaaS,North America,Enterprise,Referral,Qualified,4253,21,Won,21,4,1
1,D00002,2023-01-17,2023-01-27,rep_7,SaaS,India,Core,Referral,Closed,3905,10,Won,10,1,1
2,D00003,2023-10-29,2023-12-10,rep_5,HealthTech,APAC,Core,Inbound,Proposal,10615,42,Lost,42,4,0
3,D00004,2023-07-14,2023-08-02,rep_18,FinTech,India,Core,Partner,Negotiation,4817,19,Won,19,3,1
4,D00005,2024-02-29,2024-05-26,rep_2,HealthTech,APAC,Core,Outbound,Qualified,45203,87,Lost,87,2,0


In [22]:
df.tail()

,deal_id,created_date,closed_date,sales_rep_id,industry,region,product_type,lead_source,deal_stage,deal_amount,sales_cycle_days,outcome,total_days,close_quarter,is_won
4995,D04996,2023-10-17,2023-12-03,rep_13,Ecommerce,North America,Enterprise,Partner,Closed,2586,47,Lost,47,4,0
4996,D04997,2023-11-11,2023-12-09,rep_20,FinTech,APAC,Core,Referral,Closed,10589,28,Lost,28,4,0
4997,D04998,2023-10-19,2023-10-27,rep_24,FinTech,North America,Core,Inbound,Negotiation,57434,8,Won,8,4,1
4998,D04999,2023-03-12,2023-05-18,rep_21,SaaS,APAC,Enterprise,Inbound,Proposal,50717,67,Won,67,2,1
4999,D05000,2023-12-22,2024-03-25,rep_5,HealthTech,North America,Core,Outbound,Demo,10524,94,Won,94,1,1


In [23]:
average_win = df['is_won'].mean()
average_win * 100

45.26

In [24]:
win_by_quarter = df.groupby(df['close_quarter'])['is_won'].mean()
win_by_quarter

close_quarter
1    0.461305
2    0.445342
3    0.430622
4    0.475170
Name: is_won, dtype: float64

In [25]:
win_by_region = df.groupby(df['region'])['is_won'].mean()
win_by_region

region
APAC             0.449275
Europe           0.455799
India            0.457232
North America    0.447942
Name: is_won, dtype: float64

In [26]:
win_by_product = df.groupby(df['product_type'])['is_won'].mean()
win_by_product

product_type
Core          0.455136
Enterprise    0.449693
Pro           0.452864
Name: is_won, dtype: float64

In [27]:
deal_size_by_industry = df.groupby(df['industry'])['deal_amount'].mean()
deal_size_by_industry

industry
Ecommerce     26626.355660
EdTech        27346.715726
FinTech       25759.661686
HealthTech    25163.640594
SaaS          26502.002997
Name: deal_amount, dtype: float64

In [28]:
deal_size_by_product = df.groupby(df['product_type'])['deal_amount'].mean()
deal_size_by_product

product_type
Core          26436.738489
Enterprise    25705.325767
Pro           26699.849642
Name: deal_amount, dtype: float64

In [29]:
rep = df.groupby(df['sales_rep_id'])['deal_amount'].mean().sort_values()
rep[:5]

sales_rep_id
rep_22    21858.117925
rep_21    23265.250000
rep_24    23912.216749
rep_15    23923.657143
rep_18    24080.102151
Name: deal_amount, dtype: float64

In [30]:
stage_conversion = (df.groupby('deal_stage').agg(total_deals=('deal_id', 'count'),win_rate=('is_won', 'mean'))
      .sort_values(by='total_deals', ascending=False))

print(stage_conversion)

             total_deals  win_rate
deal_stage                        
Demo                1043  0.458293
Proposal            1009  0.446977
Closed               997  0.467402
Negotiation          995  0.466332
Qualified            956  0.422594


Part 3 – Build a Decision Engine

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [32]:
X = df[['industry', 'region', 'product_type',
        'lead_source', 'total_days',
        'deal_amount']]

y = df['is_won']

In [33]:
cat_cols = ['industry', 'region', 'product_type', 'lead_source']
num_cols = ['total_days', 'deal_amount']

In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(drop='first'), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)


In [35]:
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])


In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(drop='first'),
                                                  ['industry', 'region',
                                                   'product_type',
                                                   'lead_source']),
                                                 ('num', 'passthrough',
                                                  ['total_days',
                                                   'deal_amount'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [37]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
feature_names

array(['cat__industry_EdTech', 'cat__industry_FinTech',
       'cat__industry_HealthTech', 'cat__industry_SaaS',
       'cat__region_Europe', 'cat__region_India',
       'cat__region_North America', 'cat__product_type_Enterprise',
       'cat__product_type_Pro', 'cat__lead_source_Outbound',
       'cat__lead_source_Partner', 'cat__lead_source_Referral',
       'num__total_days', 'num__deal_amount'], dtype=object)

In [38]:
coefficients = model.named_steps['classifier'].coef_[0]
coefficients

array([ 4.95266146e-02,  1.62939629e-01,  2.43174614e-02,  8.22485393e-02,
        3.00517041e-02, -2.55999348e-02,  1.02786685e-02,  1.98086829e-02,
       -1.69024429e-02, -4.09872023e-02, -8.97084559e-02, -9.12249113e-02,
       -1.16955291e-03,  1.74195640e-06])

In [39]:
driver_df = pd.DataFrame({
    'feature': feature_names,
    'impact': coefficients
}).sort_values(by='impact')

In [40]:
driver_df

,feature,impact
11,cat__lead_source_Referral,-0.091225
10,cat__lead_source_Partner,-0.089708
9,cat__lead_source_Outbound,-0.040987
5,cat__region_India,-0.025600
8,cat__product_type_Pro,-0.016902
12,num__total_days,-0.001170
13,num__deal_amount,0.000002
6,cat__region_North America,0.010279
7,cat__product_type_Enterprise,0.019809
2,cat__industry_HealthTech,0.024317


In [44]:
predicted_prob = model.predict_proba(X)[:,1]
predicted_prob

array([0.45429608, 0.4435527 , 0.4517508 , ..., 0.5191464 , 0.48119828,
       0.42915905])

In [ ]:
forecast_df = df[df['outcome'] != 'Won'].copy()
forecast_df['predicted_win_prob'] = model.predict_proba(forecast_df[X.columns])[:, 1]
forecast_df['expected_revenue'] = (forecast_df['deal_amount'] * forecast_df['predicted_win_prob'])
forecast_df.head()


,deal_id,created_date,closed_date,sales_rep_id,industry,region,product_type,lead_source,deal_stage,deal_amount,sales_cycle_days,outcome,total_days,close_quarter,is_won,predicted_win_prob,expected_revenue
2,D00003,2023-10-29,2023-12-10,rep_5,HealthTech,APAC,Core,Inbound,Proposal,10615,42,Lost,42,4,0,0.451751,4795.334731
4,D00005,2024-02-29,2024-05-26,rep_2,HealthTech,APAC,Core,Outbound,Qualified,45203,87,Lost,87,2,0,0.443501,20047.573362
7,D00008,2024-03-07,2024-06-20,rep_3,HealthTech,APAC,Core,Referral,Proposal,8525,105,Lost,105,2,0,0.410440,3499.001156
8,D00009,2023-07-22,2023-11-19,rep_24,SaaS,APAC,Enterprise,Referral,Demo,5758,120,Lost,120,4,0,0.423903,2440.835165
12,D00013,2023-03-20,2023-05-13,rep_16,FinTech,North America,Pro,Partner,Proposal,20074,54,Lost,54,2,0,0.462852,9291.288801


Part 4 – Mini System Design

CRM (Salesforce / HubSpot)
        ->
Daily Data Extraction (ETL)
        ->
Data Warehouse (Snowflake / BigQuery / Azure SQL)
        ->
Analytics Layer (Python / ML models)
        ->
Insight Engine
        ->
Dashboard + Alerts (PowerBI / Email / Slack )

Step-by-Step Flow

CRM exports daily deal data

ETL cleans:

    Date formatting

    Missing values

    Feature engineering

Store snapshot table (daily pipeline state)

Analytics job runs:

    Recalculate win rates

    Recalculate model probabilities

    Update forecast

Alert rules check thresholds

Send alerts if triggered

Win Rate Drop Alert

Trigger:

Win rate drops >5% compared to last quarter

Alert:

"Win rate in APAC dropped from 38% to 26% in last 30 days. Primary driver: Enterprise deals."

Stagnant Pipeline Alert

Trigger:

35% deals stuck >30 days in Proposal stage

Alert:

"High proposal stagnation detected. 42% of Proposal deals older than 30 days."

Weekly Insight Summary 

Sent every Monday:

Win rate trend

Top 3 negative drivers

Top performing rep

Forecast vs target

Segment risk ranking

Part 5 – Reflection

Assumptions:

Assumption 1: CRM data is clean and reliable

Assumption 2: Historical patterns will continue

Assumption 3: Logistic regression captures all drivers

What would break in real-world production?

Data Drift, Small Sample Bias, Alert Fatigue

What would I build next if given 1 month?

Feature Enrichment, real-time Dashboard

What part of your solution are you least confident about?

Right now, the model assumes that everything that drives sales success is neatly captured in CRM fields (like deal size, industry, stage).

A basic statistical model (like logistic regression) might miss those softer, qualitative factors.